In [ ]:
```python
# Jupyter Notebook cell: Vanilla Saliency Maps for a custom ResNet50 (PyTorch)
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Configuration
num_classes = 196
weights_path = "resnet50_custom.pt"   # path to your .pt file
image_path = "example.jpg"            # path to the input image

# Model loading: ResNet50 with modified final layer
def load_model(weights_path, num_classes, device):
    model = models.resnet50(pretrained=False)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    # Robust loading for different checkpoint formats
    ckpt = torch.load(weights_path, map_location=device)
    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        state = ckpt["state_dict"]
    elif isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        state = ckpt["model_state_dict"]
    else:
        state = ckpt
    # strip possible "module." prefixes from DataParallel
    new_state = {}
    for k, v in state.items():
        new_key = k.replace("module.", "") if k.startswith("module.") else k
        new_state[new_key] = v
    model.load_state_dict(new_state, strict=True)
    model.to(device).eval()
    return model

# Preprocessing: ImageNet evaluation transforms
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

# Utility to prepare a displayable (unnormalized) image resized to 224
def open_image_for_display(path):
    img = Image.open(path).convert("RGB")
    img_disp = img.resize((224, 224))
    return img, img_disp

# Saliency logic
def compute_saliency(image_path, model, transform, device):
    orig_img, display_img = open_image_for_display(image_path)
    input_tensor = transform(orig_img).unsqueeze(0).to(device)
    input_tensor.requires_grad_()
    model.zero_grad()
    outputs = model(input_tensor)            # logits
    top_idx = outputs.argmax(dim=1).item()
    score = outputs[0, top_idx]
    score.backward()
    saliency = input_tensor.grad.data.abs().squeeze(0)   # (C, H, W)
    saliency, _ = torch.max(saliency, dim=0)           # (H, W)
    saliency = saliency.cpu().numpy()
    # Normalize to [0,1]
    saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)
    return display_img, saliency, top_idx

# Visualization: original and saliency heatmap side-by-side
def plot_saliency(display_img, saliency, cmap="hot"):
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(display_img)
    axes[0].axis("off")
    axes[0].set_title("Original")
    im = axes[1].imshow(saliency, cmap=cmap)
    axes[1].axis("off")
    axes[1].set_title("Vanilla Saliency Map")
    fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

# Example usage
model = load_model(weights_path, num_classes, device)
display_img, saliency_map, predicted_class = compute_saliency(image_path, model, val_transform, device)
plot_saliency(display_img, saliency_map)
print("Predicted class index:", predicted_class)
```